# 04 — Réseaux bayéniens à partir des sorties MALT

## 1 — Objectif

Ce notebook construit des variables **binaires au niveau accident** (présence de motifs `z` au-dessus d’un seuil de confiance), puis apprend des **réseaux bayéniens discrets** avec **pgmpy** (score **BIC**, recherche **HillClimbing** sous contraintes d’ordre macro).

### Nuances d’interprétation

- **\(X_{i,k}=0\)** : le motif \(k\) n’est **pas** identifié dans le récit de l’accident \(i\) au-dessus du seuil — ce n’est **pas** une preuve d’absence physique du facteur.
- Les **arcs** du BN encodent des **dépendances conditionnelles** apprises (avec estimateur bayésien ou MLE), **pas** une causalité démontrée.


## 2 — Rappels formels (Markdown + LaTeX)

### Agrégation accident × motif

Pour un accident \(i\) et un topic \(k\) retenu après filtrage support :

\[
X_{i,k} = \mathbf{1}\left\{ \exists \text{ unité } j \text{ de } i : \hat{z}_j = k,\; \text{conf}(j) \ge \tau \right\}
\]

### Factorisation du BN

\[
P(\mathbf{X}) = \prod_{v \in \mathcal{V}} P\left(X_v \mid \mathrm{Pa}(v)\right)
\]

### BIC (structure)

\[
\mathrm{BIC} = \log \hat{L}(\mathcal{G}, \mathcal{D}) - \frac{d}{2}\log n
\]

### Lift binaire (parent → enfant)

\[
\mathrm{Lift}(Y\mid X) = \frac{P(Y{=}1 \mid X{=}1)}{P(Y{=}1)}
\]

### Sparsité (densité du graphe)

Proportion d’arcs présents parmi les couples ordonnés de nœuds (hors boucles).


In [2]:
# --- Paramètres (papermill : `papermill ... -p KEY valeur`) ---
MALT_EXPORTS_DIR = "runs/malt_btp_to_mettalurgie_qwen06/exports"
OUTPUT_DIR = "outputs/bn_malt"
CONFIDENCE_THRESHOLD = 0.50
MIN_TOPIC_ACCIDENT_SUPPORT = 20
MAX_TOPICS_PER_MACRO = 6
INCLUDE_MACRO_NODES = True
INCLUDE_SEVERITY = False
THEMES_OPENAI_CSV = ""
LEARN_UNCONSTRAINED_TOPIC = True
MAX_INDEGREE = 3
EQUIVALENT_SAMPLE_SIZE = 5
RANDOM_SEED = 42
WARN_MAX_BINARY_NODES = 30


In [3]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np

if int(np.__version__.split(".", 1)[0]) >= 2:
    _py = sys.executable
    raise ImportError(
        "NumPy 2.x est chargé alors que le matplotlib de cet environnement (Anaconda) "
        "est souvent compilé pour NumPy 1.x → conflit à l’import de matplotlib.\n\n"
        f"Interpréteur du noyau : {_py}\n\n"
        "Corrigez puis Kernel → Restart, par exemple :\n\n"
        f'  "{_py}" -m pip install "numpy<2" --force-reinstall\n\n'
        "ou :\n\n"
        '  conda install "numpy<2" "matplotlib" "scipy" -y\n\n'
        "Réinstallez aussi les deps du dépôt : pip install -r requirements.txt"
    )

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def _find_repo_root() -> Path:
    # Racine du dépôt (dossier contenant topic_eval/) sans importer topic_eval.
    candidates: list[Path] = []
    try:
        candidates.append(Path(__file__).resolve().parent)
    except NameError:
        pass
    candidates.append(Path.cwd().resolve())
    seen: set[Path] = set()
    for start in candidates:
        for p in [start, *start.parents]:
            if p in seen:
                continue
            seen.add(p)
            if (p / "topic_eval" / "__init__.py").is_file():
                return p
    return Path.cwd().resolve()


REPO = _find_repo_root()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from topic_eval.paths import resolve_repo_path

EXPORTS = resolve_repo_path(MALT_EXPORTS_DIR, REPO)
OUT_ROOT = resolve_repo_path(OUTPUT_DIR, REPO)
TABLES = OUT_ROOT / "tables"
FIGURES = OUT_ROOT / "figures"
MODELS = OUT_ROOT / "models"
REPORTS = OUT_ROOT / "reports"

from bn_malt.utils import ensure_output_dirs, load_metadata_for_bn
from bn_malt.aggregate_malt_variables import create_accident_topic_matrix, export_aggregate_outputs
from bn_malt.bn_structure import (
    build_blacklist,
    export_edge_tables,
    learn_macro_constrained_structure,
    learn_unconstrained_structure,
    macro_chain_model,
)
from bn_malt.bn_learning import (
    drop_constant_columns,
    export_cpds_to_dir,
    fit_bn_parameters,
    save_bn_pickle,
    try_write_bif,
)
from bn_malt.bn_inference import conditional_prob_table, run_bn_queries
from bn_malt.bn_visualization import (
    build_topic_node_label_map,
    plot_adjacency_heatmap,
    plot_bn_graph,
    try_plotly_interactive,
)
from bn_malt.scenario_mining import export_scenarios, extract_typical_scenarios
from bn_malt.bn_diagnostics import compare_structure_rows, run_model_diagnostics
from bn_malt.reporting import write_bn_malt_report

try:
    import pgmpy  # noqa: F401
except ImportError as _e:
    raise ImportError(
        "Le package « pgmpy » n’est pas installé pour l’interpréteur de ce noyau Jupyter.\n\n"
        f"  Interpréteur : {sys.executable}\n\n"
        "Installez-le (même environnement que le noyau), puis Kernel → Restart :\n\n"
        f"  {sys.executable} -m pip install \"pgmpy>=0.1.23,<1.0\" \"numpy<2\"\n\n"
        "ou, à la racine du dépôt :\n\n"
        "  pip install -r requirements.txt\n"
    ) from _e

ensure_output_dirs(OUT_ROOT)
np.random.seed(int(RANDOM_SEED))
warnings.filterwarnings("ignore", category=UserWarning)
print("REPO =", REPO)
print("EXPORTS =", EXPORTS)
print("OUT_ROOT =", OUT_ROOT)


REPO = C:\Users\aho\Documents\analysis factor project\SCGM\SCGM
EXPORTS = C:\Users\aho\Documents\analysis factor project\SCGM\SCGM\runs\malt_btp_to_mettalurgie_qwen06\exports
OUT_ROOT = C:\Users\aho\Documents\analysis factor project\SCGM\SCGM\outputs\bn_malt


## 3 — Chargement des exports MALT et EDA rapide


In [4]:
meta, exports_path = load_metadata_for_bn(MALT_EXPORTS_DIR, repo_root=REPO)
prob_y_z = np.load(exports_path / "pt_y_given_z.npy")

print(meta.shape)
display(meta.head(3))

_has_sev_panel = bool(INCLUDE_SEVERITY) and "pred_severity" in meta.columns
_nc = 2 if _has_sev_panel else 1
fig, axes = plt.subplots(1, _nc, figsize=(10 if _has_sev_panel else 6, 4))
if _nc == 1:
    axes = [axes]
if "z_confidence" in meta.columns:
    sns.histplot(meta["z_confidence"].astype(float), bins=40, ax=axes[0])
    axes[0].axvline(float(CONFIDENCE_THRESHOLD), color="red", ls="--", label="τ")
    axes[0].legend()
    axes[0].set_title("Confiance z (MALT)")
if _has_sev_panel:
    meta["pred_severity"].astype(str).value_counts().head(8).plot.bar(ax=axes[1])
    axes[1].set_title("Gravité prédite (unités)")
plt.tight_layout()
p = FIGURES / "eda_malt_metadata.png"
plt.savefig(p, dpi=150, bbox_inches="tight")
plt.close()
print("Figure :", p)


(22487, 21)


,accident_id,fact_id,sentence,accident_summary,pred_label,pred_subtype,pred_severity,doc_id,p0_macro_name,pt_macro_name,...,z_confidence,z_dominant_macro,p_A0_given_z,p_A1_given_z,p_B_given_z,p_C_given_z,pt_A0,pt_A1,pt_B,pt_C
0,A27B5D1A83123AD6C1258D7A003AD5DE,1,"Un responsable bureau détude et marketing, âg...","Un responsable bureau détude et marketing, âg...",A0,A0_TASK_ACTIVITY,NONE,1,A0,A0,...,0.369107,A0,0.601551,0.150499,0.165933,0.082017,0.520305,0.177410,0.178035,0.124251
1,A27B5D1A83123AD6C1258D7A003AD5DE,2,Il a chuté sur la voie publique.,"Un responsable bureau détude et marketing, âg...",C,C_UNSPECIFIED_INJURY,NONE,2,B,B,...,0.223329,B,0.067417,0.149023,0.626664,0.156896,0.136696,0.181675,0.472634,0.208995
2,A27B5D1A83123AD6C1258D7A003AD5DE,3,Les secours ont été engagés à 8h33.,"Un responsable bureau détude et marketing, âg...",A0,A0_OTHER,NONE,3,A0,A0,...,0.438746,C,0.199345,0.129104,0.271948,0.399603,0.303399,0.165291,0.246271,0.285040


Figure : C:\Users\aho\Documents\analysis factor project\SCGM\SCGM\outputs\bn_malt\figures\eda_malt_metadata.png


## 4 — Agrégation accident × topics (variables binaires `Z_*` et `M_*`)


In [5]:
acc_df, sel, map_df = create_accident_topic_matrix(
    meta,
    accident_id_col="accident_id",
    z_col="z_hat",
    z_conf_col="z_confidence",
    z_macro_col="z_dominant_macro",
    confidence_threshold=float(CONFIDENCE_THRESHOLD),
    min_topic_accident_support=int(MIN_TOPIC_ACCIDENT_SUPPORT),
    max_topics_per_macro=int(MAX_TOPICS_PER_MACRO),
    prob_y_z=prob_y_z,
    include_macro_aggregate_nodes=bool(INCLUDE_MACRO_NODES),
    include_severity=bool(INCLUDE_SEVERITY),
    severity_col="pred_severity",
    warn_max_binary_nodes=int(WARN_MAX_BINARY_NODES),
)
export_aggregate_outputs(acc_df, sel, map_df, TABLES)
print(acc_df.shape)
display(sel.head(10))


(3212, 15)


,variable,z_id,macro,n_accidents_with_topic,share_accidents
0,Z_02_C,2,C,119,0.037049
1,Z_06_C,6,C,27,0.008406
2,Z_11_C,11,C,488,0.151930
3,Z_13_C,13,C,56,0.017435
4,Z_14_C,14,C,239,0.074408
5,Z_15_A0,15,A0,67,0.020859
6,Z_22_B,22,B,23,0.007161
7,Z_27_B,27,B,98,0.030511
8,Z_29_C,29,C,842,0.262142
9,Z_31_A1,31,A1,126,0.039228


## 5 — EDA de la matrice accident × variables retenues


In [6]:
topic_cols = [c for c in acc_df.columns if str(c).startswith("Z_")]
macro_cols = [c for c in acc_df.columns if str(c).startswith("M_")]
print("n topics:", len(topic_cols), "n macros:", len(macro_cols))

if topic_cols:
    M = acc_df[topic_cols].to_numpy(dtype=float)
    share = float(M.mean())
    print("Part moyenne de 1 (topics) :", round(share, 4))
    plt.figure(figsize=(10, 6))
    co = np.corrcoef(M.T)
    sns.heatmap(co, xticklabels=False, yticklabels=False, cmap="vlag", center=0)
    plt.title("Corrélations entre colonnes Z (aperçu)")
    p = FIGURES / "topic_correlation_heatmap.png"
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close()
    print("Figure :", p)


n topics: 10 n macros: 4
Part moyenne de 1 (topics) : 0.0649
Figure : C:\Users\aho\Documents\analysis factor project\SCGM\SCGM\outputs\bn_malt\figures\topic_correlation_heatmap.png


## 6 — Contraintes de structure (ordre macro)

On impose une **blacklist** d’arcs interdits : aucun parent ne peut être « après » son enfant dans l’ordre
**A0 → A1 → B → C** (les nœuds hors cette chaîne, s’il en existe, sont positionnés sans colonne dédiée supplémentaire).

Les fichiers `forbidden_edges.csv` et `allowed_edges.csv` documentent ces ensembles (indices + arcs appris).


## 7 — Apprentissage : BN **macro** (chaîne fixe des agrégats `M_*`)


In [7]:
from pgmpy.models import BayesianNetwork

macro_var_map = {f"M_{m}": m for m in ("A0", "A1", "B", "C")}
if INCLUDE_SEVERITY and "Severity_high" in acc_df.columns:
    macro_var_map["Severity_high"] = "Severity"
    macro_tpl, macro_edges_tpl = macro_chain_model("Severity_high")
else:
    macro_edges_tpl = [("M_A0", "M_A1"), ("M_A1", "M_B"), ("M_B", "M_C")]
    macro_tpl = BayesianNetwork(macro_edges_tpl)

macro_node_list = [n for n in macro_tpl.nodes() if n in acc_df.columns]
macro_data = acc_df[macro_node_list].copy()
macro_data, macro_used = drop_constant_columns(macro_data, list(macro_data.columns))
macro_edges_sub = [(u, v) for (u, v) in macro_tpl.edges() if u in macro_used and v in macro_used]
macro_model = BayesianNetwork(macro_edges_sub)
macro_edges_export = list(macro_edges_sub)
macro_model = fit_bn_parameters(
    macro_model,
    macro_data,
    estimator="bayesian",
    equivalent_sample_size=int(EQUIVALENT_SAMPLE_SIZE),
)
save_bn_pickle(macro_model, MODELS / "bn_macro_chain.pkl")
export_cpds_to_dir(macro_model, TABLES / "cpds_macro", prefix="macro")
try_write_bif(macro_model, MODELS / "bn_macro_chain.bif")
print("Macro BN — nœuds:", list(macro_model.nodes()))


C:\Users\aho\Documents\analysis factor project\SCGM\SCGM\.venv\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.13). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\aho\Documents\analysis factor project\SCGM\SCGM\.venv\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
C:\Users\aho\Documents\analysis factor project\SCGM\SCGM\.venv\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a

Macro BN — nœuds: ['M_A0', 'M_A1', 'M_B', 'M_C']


## 8 — Apprentissage : BN **topics** sous contraintes (HillClimb + BIC)


In [11]:
topic_var_map = {str(r["variable"]): str(r["macro"]) for _, r in sel.iterrows()}
if INCLUDE_SEVERITY and "Severity_high" in acc_df.columns:
    topic_var_map["Severity_high"] = "Severity"

topic_node_list = topic_cols.copy()
if INCLUDE_SEVERITY and "Severity_high" in acc_df.columns:
    topic_node_list = topic_node_list + ["Severity_high"]

topic_data = acc_df[[c for c in topic_node_list if c in acc_df.columns]].copy()
topic_data, topic_used = drop_constant_columns(topic_data, list(topic_data.columns))
topic_var_map_f = {k: v for k, v in topic_var_map.items() if k in topic_used}

topic_model, topic_edges = learn_macro_constrained_structure(
    topic_data,
    topic_var_map_f,
    max_indegree=int(MAX_INDEGREE),
)
topic_model = fit_bn_parameters(
    topic_model,
    topic_data,
    estimator="bayesian",
    equivalent_sample_size=int(EQUIVALENT_SAMPLE_SIZE),
)
save_bn_pickle(topic_model, MODELS / "bn_topic_constrained.pkl")
export_cpds_to_dir(topic_model, TABLES / "cpds_topic", prefix="topic")
try_write_bif(topic_model, MODELS / "bn_topic_constrained.bif")

bl_topic = build_blacklist(list(topic_used), topic_var_map_f)
allowed_hint = sorted(set(topic_edges) | set(macro_edges_export))
export_edge_tables(macro_edges_export, topic_edges, bl_topic, allowed_hint, TABLES)
print("Topic BN (contraint) —", topic_model.number_of_nodes(), "nœuds,", len(topic_edges), "arcs")


  0%|          | 0/1000000 [00:00<?, ?it/s]

Topic BN (contraint) — 8 nœuds, 7 arcs


## 9 — Diagnostics (`check_model`, DAG, degrés)


In [12]:
diag_rows = [
    run_model_diagnostics(macro_model, "macro_chain"),
    run_model_diagnostics(topic_model, "topic_constrained"),
]
diag_df = pd.DataFrame(diag_rows)
display(diag_df)


,model_name,n_nodes,n_edges,acyclic,check_model_ok,check_model_error,max_indegree,mean_degree,n_weakly_connected,n_isolated,isolated_nodes
0,macro_chain,4,3,True,True,,1,1.50,1,0,
1,topic_constrained,8,7,True,True,,2,1.75,2,0,


## 10 — Visualisation du graphe et heatmap d’adjacence


In [13]:
from pathlib import Path

if str(THEMES_OPENAI_CSV).strip():
    _tp = Path(str(THEMES_OPENAI_CSV).strip()).expanduser()
    if not _tp.is_absolute():
        _tp = resolve_repo_path(str(_tp), REPO)
    _themes_path = _tp if _tp.is_file() else None
else:
    _themes_path = None
    for _c in (
        EXPORTS / "themes_by_z_openai_malt.csv",
        EXPORTS / "themes_by_z_openai.csv",
        EXPORTS / "themes_by_z_malt.csv",
    ):
        if _c.is_file():
            _themes_path = _c
            break

if _themes_path is not None:
    themes_df = pd.read_csv(_themes_path)
    print("Fichier libellés thèmes :", _themes_path)
else:
    themes_df = pd.DataFrame()
    print("Aucun CSV thèmes trouvé ; libellés de repli sur les codes Z_*.")

node_label_map = build_topic_node_label_map(list(topic_model.nodes()), themes_df, wrap_width=32)
plot_bn_graph(
    topic_model,
    topic_var_map_f,
    FIGURES / "bn_topic_constrained.png",
    title="Réseau bayésien — motifs (structure contrainte)",
    node_label_map=node_label_map,
)
plot_adjacency_heatmap(
    topic_model,
    list(topic_used),
    FIGURES / "bn_topic_adjacency.png",
    title="Adjacence — BN topics",
)
ok_html = try_plotly_interactive(
    topic_model,
    FIGURES / "bn_topic_interactive.html",
    node_label_map=node_label_map,
    title="Réseau bayésien — exploration interactive",
)
print("Plotly HTML :", ok_html)


Fichier libellés thèmes : C:\Users\aho\Documents\analysis factor project\SCGM\SCGM\runs\malt_btp_to_mettalurgie_qwen06\exports\themes_by_z_openai_malt.csv
Plotly HTML : True


## 11 — Inférence (VariableElimination) et lifts sur les arcs


In [14]:
q_df, lift_df = run_bn_queries(topic_model)
q_df.to_csv(TABLES / "query_results.csv", index=False)
lift_df.to_csv(TABLES / "lift_results.csv", index=False)
display(lift_df.head(15))

# Exemple A1 → B sur variables macro agrégées si présentes
if "M_A1" in topic_model.nodes() and "M_B" in topic_model.nodes():
    cpt = conditional_prob_table(topic_model, "M_B", "M_A1")
    cpt.to_csv(TABLES / "conditional_M_B_given_M_A1.csv", index=False)
    display(cpt)


,parent,child,p_y1,p_y1_given_x1,lift
0,Z_27_B,Z_11_C,0.152471,0.360697,2.365669
1,Z_29_C,Z_11_C,0.152471,0.021610,0.141734
2,Z_11_C,Z_14_C,0.075070,0.022936,0.305525
3,Z_11_C,Z_02_C,0.037768,0.063710,1.686886
4,Z_15_A0,Z_13_C,0.018185,0.161871,8.901494
5,Z_27_B,Z_29_C,0.262512,0.052239,0.198996
6,Z_29_C,Z_06_C,0.009170,0.001480,0.161413


## 12 — Configurations typiques de co-présence de motifs


In [15]:
_sev_col = None
if bool(INCLUDE_SEVERITY):
    if "Severity_high" in acc_df.columns:
        _sev_col = "Severity_high"
    elif "Severity_ord" in acc_df.columns:
        _sev_col = "Severity_ord"

freq_df, high_df = extract_typical_scenarios(
    acc_df,
    topic_model,
    topic_cols,
    accident_id_col="accident_id",
    severity_high_col=_sev_col,
    min_support=5,
    top_n=30,
    metadata_unit=meta,
    text_col="sentence" if "sentence" in meta.columns else "accident_summary",
)
export_scenarios(freq_df, high_df, TABLES)
display(freq_df.head(8))
if len(high_df):
    display(high_df.head(8))
else:
    print("Pas d’export « risque gravité » (mode sans colonne de gravité ou effectifs nuls).")


,scenario_id,macro_path,topics_present,support,representative_accidents,representative_sentences
0,0,,,1468,000D84B1F9D28BF4C125719D005193F7 | 00191D8E37D...,Un salarié de 31 ans est décédé. || Les causes...
1,1,C,Z_29_C,672,0017468054888DC0C125719D005572CD | 00A39950955...,Un responsable maintenance de 28 ans travaille...
2,2,C,Z_11_C,371,00C342AB6F87A9F9C125719D0054ED71 | 00CF4055BA2...,"La victime, âgée de 37 ans, est magasinier dan..."
3,3,C,Z_14_C,140,002F36405C99A76CC12586E20046C8C9 | 024C9784FD3...,Dans le cadre de la rénovation d'un bâtiment i...
4,4,C -> C,Z_14_C + Z_29_C,66,039B4B557BB3E470C125719D00564BF0 | 043127691E5...,"La victime - 24 ans, chaudronnier - est affect..."
5,5,A1,Z_31_A1,60,069FE54DF809CADEC12579E4004577D5 | 0977CE3CA55...,"L'entreprise de carrosserie a fait réaliser, s..."
6,6,C,Z_02_C,58,06BCB1038E4C6C97C125719D0054FD0B | 07EFA6F4C39...,"La victime, 37 ans, monteur, travaillait à l'i..."
7,7,B,Z_27_B,49,005CDF5BCF99D27FC1258D1C0021FEDC | 0668C9422A6...,Un opérateur de 60 ans se rendait au travail a...


Pas d’export « risque gravité » (mode sans colonne de gravité ou effectifs nuls).


## 13 — Helpers d’affichage (résumé nœud, scénario)


In [16]:
def display_bn_node_summary(model, node: str, max_lines: int = 40) -> None:
    for cpd in model.get_cpds():
        if cpd.variable == node:
            txt = str(cpd)
            lines = txt.splitlines()
            print("\n".join(lines[:max_lines]))
            if len(lines) > max_lines:
                print("…")
            return
    print("CPD introuvable pour", node)


def display_scenario(row: pd.Series) -> None:
    keys = [
        "scenario_id",
        "macro_path",
        "topics_present",
        "support",
        "representative_accidents",
        "representative_sentences",
    ]
    for k in keys:
        if k in row.index:
            print(f"{k}: {row[k]}")


if len(sel):
    display_bn_node_summary(topic_model, str(sel.iloc[0]["variable"]))
if len(freq_df):
    display_scenario(freq_df.iloc[0])


+-----------+----------------------+-------------------+
| Z_11_C    | Z_11_C(0)            | Z_11_C(1)         |
+-----------+----------------------+-------------------+
| Z_02_C(0) | 0.9668989547038328   | 0.936289500509684 |
+-----------+----------------------+-------------------+
| Z_02_C(1) | 0.033101045296167246 | 0.063710499490316 |
+-----------+----------------------+-------------------+
scenario_id: 0
macro_path: 
topics_present: 
support: 1468
representative_accidents: 000D84B1F9D28BF4C125719D005193F7 | 00191D8E37D5D3AFC125719D0050EF02 | 001C521AF6A9CD1CC125719D0055BEF1 | 001ED9E54DAF8A72C1257D1700269769 | 0025DCE09D0C01F3C125719D005066E6
representative_sentences: Un salarié de 31 ans est décédé. || Les causes et les circonstances de l'accident ne sont pas connues. || La victime - 17 ans, apprenti électromécanicien, en contrat d'apprentissage - était occupée à baguer un pignon sur une machine de fabrication maison. || Cette machine se compose d'un vérin pneumatique disposé su

## 14 — Comparaison de structures (chaîne macro vs topic contraint vs topic sans contrainte)

Métriques : nombre d’arcs, violations de la blacklist macro (pour le BN non contraint), densité, nœuds isolés.
Le score BIC global pgmpy n’est pas toujours comparable entre structures différentes ; on privilégie ces indicateurs de complexité et de respect des contraintes.


In [17]:
topic_unc_model = None
topic_unc_edges: list = []
if bool(LEARN_UNCONSTRAINED_TOPIC):
    topic_unc_model, topic_unc_edges = learn_unconstrained_structure(
        topic_data,
        list(topic_used),
        max_indegree=int(MAX_INDEGREE),
    )
    topic_unc_model = fit_bn_parameters(
        topic_unc_model,
        topic_data,
        estimator="bayesian",
        equivalent_sample_size=int(EQUIVALENT_SAMPLE_SIZE),
    )
    save_bn_pickle(topic_unc_model, MODELS / "bn_topic_unconstrained.pkl")

diag_rows = [
    run_model_diagnostics(macro_model, "macro_chain"),
    run_model_diagnostics(topic_model, "topic_constrained"),
]
if topic_unc_model is not None:
    diag_rows.append(run_model_diagnostics(topic_unc_model, "topic_unconstrained"))
diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(TABLES / "bn_model_diagnostics.csv", index=False)

comp_rows = []
comp_rows.append(compare_structure_rows("macro_chain", macro_model, macro_var_map))
comp_rows.append(compare_structure_rows("topic_constrained", topic_model, topic_var_map_f))
if topic_unc_model is not None:
    comp_rows.append(compare_structure_rows("topic_unconstrained", topic_unc_model, topic_var_map_f))
comp_df = pd.DataFrame(comp_rows)
comp_df.to_csv(TABLES / "bn_structure_comparison.csv", index=False)
display(comp_df)


  0%|          | 0/1000000 [00:00<?, ?it/s]

,structure,n_arcs,n_nodes,macro_order_violations,density,n_isolated
0,macro_chain,3,4,0,0.250000,0
1,topic_constrained,7,8,0,0.125000,0
2,topic_unconstrained,8,8,2,0.142857,0


## 15 — Rapport Markdown synthétique


In [18]:
params = {
    "CONFIDENCE_THRESHOLD": CONFIDENCE_THRESHOLD,
    "MIN_TOPIC_ACCIDENT_SUPPORT": MIN_TOPIC_ACCIDENT_SUPPORT,
    "MAX_TOPICS_PER_MACRO": MAX_TOPICS_PER_MACRO,
}
write_bn_malt_report(
    REPORTS / "bn_malt_summary.md",
    n_accidents=int(acc_df.shape[0]),
    n_topics_selected=int(len(sel)),
    params=params,
    diagnostics_df=diag_df,
    comparison_df=comp_df,
    figure_paths=[
        str(FIGURES / "eda_malt_metadata.png"),
        str(FIGURES / "bn_topic_constrained.png"),
        str(FIGURES / "bn_topic_adjacency.png"),
    ],
)
print("Rapport :", REPORTS / "bn_malt_summary.md")


Rapport : C:\Users\aho\Documents\analysis factor project\SCGM\SCGM\outputs\bn_malt\reports\bn_malt_summary.md


## 16 — Export LaTeX (tableaux diagnostics / comparaison)


In [19]:
tex_diag = REPORTS / "bn_model_diagnostics.tex"
tex_comp = REPORTS / "bn_structure_comparison.tex"
diag_df.to_latex(tex_diag, index=False)
comp_df.to_latex(tex_comp, index=False)
print(tex_diag, tex_comp)


C:\Users\aho\Documents\analysis factor project\SCGM\SCGM\outputs\bn_malt\reports\bn_model_diagnostics.tex C:\Users\aho\Documents\analysis factor project\SCGM\SCGM\outputs\bn_malt\reports\bn_structure_comparison.tex
